# E1 — Most Popular Baseline

## Đang làm gì?
Most Popular **không train model**. Notebook đếm độ phổ biến của item từ **train interactions**, rồi với mỗi test user:
1. loại item đã xuất hiện trong train của user đó;
2. lấy Top-20 item phổ biến nhất còn lại;
3. so với test interactions để tính Recall/NDCG.

## Được gì?
- `topk_test.npz`: Top-K recommendation của từng test user.
- `metrics.json`: Recall/NDCG của Most Popular.
- `export_meta.json`: metadata của lần chạy.
- `verification.json`: kiểm tra output/protocol.

## Để làm gì?
Most Popular là **mốc tham chiếu thuần popularity** để so với LightGCN và RLMRec-Con. Sau này E2 dùng chính Top-K này để tính ARP/APLT, Head/Tail Recall và phân tích popularity bias.


In [ ]:
# ==========================================
# CELL 1 — CONFIGURATION & ENVIRONMENT
# ==========================================

import sys, json, pickle
import numpy as np
import torch
from pathlib import Path
from google.colab import drive

K = 20
DATASET = "amazon"

print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/Data_SNA_Group14")
DATA_DIR = ROOT / "data/raw/amazon"
OUTPUT_DIR = ROOT / "experiments/E1/most_popular"
PRED_DIR = ROOT / "experiments/predictions/most_popular"

for filename in ["trn_mat.pkl", "val_mat.pkl", "tst_mat.pkl"]:
    assert (DATA_DIR / filename).is_file(), f"Missing dataset file: {filename}"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATASET)
print("K:", K)
print("Data:", DATA_DIR)
print("Metrics output:", OUTPUT_DIR)
print("Predictions output:", PRED_DIR)
print("\nCELL 1: SUCCESS")


In [ ]:
# ==========================================
# CELL 2 — LOAD DATA & BUILD POPULARITY RANKING
# ==========================================

import scipy.sparse as sp

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

trn_mat = load_pickle(DATA_DIR / "trn_mat.pkl").tocsr()
val_mat = load_pickle(DATA_DIR / "val_mat.pkl").tocsr()
tst_mat = load_pickle(DATA_DIR / "tst_mat.pkl").tocsr()

assert sp.issparse(trn_mat) and sp.issparse(val_mat) and sp.issparse(tst_mat)
assert trn_mat.shape == val_mat.shape == tst_mat.shape

user_num, item_num = trn_mat.shape

# Popularity = số train interactions của từng item
item_degree = np.asarray(trn_mat.sum(axis=0)).ravel().astype(np.int64)
item_ids = np.arange(item_num, dtype=np.int64)

# degree giảm dần; nếu bằng nhau thì item_id tăng dần
ranked_by_popularity = np.lexsort((item_ids, -item_degree))

test_counts = np.diff(tst_mat.indptr)
test_users = np.where(test_counts > 0)[0].astype(np.int64)

print("Users:", user_num)
print("Items:", item_num)
print("Train interactions:", trn_mat.nnz)
print("Validation interactions:", val_mat.nnz)
print("Test interactions:", tst_mat.nnz)
print("Test users:", len(test_users))

print("\nTop-10 items by train degree:")
for r, iid in enumerate(ranked_by_popularity[:10], 1):
    print(f"{r:2d}. item={int(iid):4d} | degree={int(item_degree[iid])}")

print("\nCELL 2: SUCCESS")


In [ ]:
# ==========================================
# CELL 3 — GENERATE MOST POPULAR TOP-K
# ==========================================

topk_items = np.empty((len(test_users), K), dtype=np.int32)
topk_scores = np.empty((len(test_users), K), dtype=np.float32)

ground_truth = []

for row, uid in enumerate(test_users):
    # Train items của user này
    s, e = trn_mat.indptr[uid], trn_mat.indptr[uid + 1]
    seen_train = trn_mat.indices[s:e]

    # Giữ đúng protocol test hiện tại: mask train, không mask validation
    seen_mask = np.zeros(item_num, dtype=bool)
    seen_mask[seen_train] = True
    candidates = ranked_by_popularity[~seen_mask[ranked_by_popularity]]

    assert len(candidates) >= K
    recs = candidates[:K]
    topk_items[row] = recs.astype(np.int32)
    topk_scores[row] = item_degree[recs].astype(np.float32)

    ts, te = tst_mat.indptr[uid], tst_mat.indptr[uid + 1]
    ground_truth.append(tst_mat.indices[ts:te].tolist())

# Strong guard: không item train nào được lọt vào Top-K
for row, uid in enumerate(test_users):
    s, e = trn_mat.indptr[uid], trn_mat.indptr[uid + 1]
    seen = set(trn_mat.indices[s:e].tolist())
    assert not any(int(i) in seen for i in topk_items[row])

print("Top-K generated for", len(test_users), "test users")
print("item_ids shape:", topk_items.shape)
print("scores shape:", topk_scores.shape)
print("\nCELL 3: SUCCESS")


In [ ]:
# ==========================================
# CELL 4 — EVALUATE RECALL/NDCG
# Same binary-relevance Top-K definitions used by this project
# ==========================================

def recall_at_k(recs, truth, k):
    truth = set(map(int, truth))
    hits = sum(int(i) in truth for i in recs[:k])
    return hits / len(truth) if truth else 0.0

def ndcg_at_k(recs, truth, k):
    truth = set(map(int, truth))
    if not truth:
        return 0.0
    dcg = sum(
        (1.0 / np.log2(rank + 2))
        for rank, iid in enumerate(recs[:k])
        if int(iid) in truth
    )
    ideal = min(len(truth), k)
    idcg = sum(1.0 / np.log2(rank + 2) for rank in range(ideal))
    return dcg / idcg if idcg > 0 else 0.0

metrics = {}
for k in [10, 20]:
    recalls = [recall_at_k(topk_items[r], ground_truth[r], k) for r in range(len(test_users))]
    ndcgs = [ndcg_at_k(topk_items[r], ground_truth[r], k) for r in range(len(test_users))]
    metrics[f"recall@{k}"] = float(np.mean(recalls))
    metrics[f"ndcg@{k}"] = float(np.mean(ndcgs))

print("=== MOST POPULAR TEST METRICS ===")
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")

print("\nCELL 4: SUCCESS")


In [ ]:
# ==========================================
# CELL 5 — SAVE OUTPUTS & VERIFY
# ==========================================

from datetime import datetime, timezone

TOPK_FILE = PRED_DIR / "topk_test.npz"
METRICS_FILE = OUTPUT_DIR / "metrics.json"
META_FILE = PRED_DIR / "export_meta.json"
VERIFICATION_FILE = PRED_DIR / "verification.json"

# Không ghi đè kết quả cũ một cách im lặng
for path in [TOPK_FILE, METRICS_FILE, META_FILE, VERIFICATION_FILE]:
    if path.exists():
        raise FileExistsError(f"{path} already exists. Move/archive it before rerunning.")

np.savez_compressed(
    TOPK_FILE,
    user_ids=test_users.astype(np.int64),
    item_ids=topk_items.astype(np.int32),
    scores=topk_scores.astype(np.float32),
)

METRICS_FILE.write_text(json.dumps({
    "experiment": "E1_Most_Popular",
    "model": "most_popular",
    "dataset": DATASET,
    "top_k": K,
    "test_metrics": metrics,
}, indent=2, ensure_ascii=False), encoding="utf-8")

META_FILE.write_text(json.dumps({
    "model": "most_popular",
    "dataset": DATASET,
    "split": "test",
    "top_k": K,
    "test_user_count": int(len(test_users)),
    "total_users": int(user_num),
    "total_items": int(item_num),
    "popularity_definition": "item degree in train interaction matrix",
    "ranking_candidate_pool": "all items except items observed in train for each user",
    "masked_splits": ["train"],
    "validation_masked": False,
    "test_labels_used_to_rank": False,
    "tie_break": "item_id ascending when train degree is equal",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2, ensure_ascii=False), encoding="utf-8")

saved = np.load(TOPK_FILE)
assert saved["user_ids"].shape == (len(test_users),)
assert saved["item_ids"].shape == (len(test_users), K)
assert saved["scores"].shape == (len(test_users), K)
assert np.isfinite(saved["scores"]).all()

VERIFICATION_FILE.write_text(json.dumps({
    "status": "passed",
    "model": "most_popular",
    "top_k": K,
    "excluded_train_items": True,
    "test_users_unique": bool(len(np.unique(test_users)) == len(test_users)),
    "output_shapes_valid": True,
    "test_metrics": metrics,
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("=== MOST POPULAR BASELINE COMPLETE ===")
print("Verification: passed")
print("Metrics:", METRICS_FILE)
print("Top-K:", TOPK_FILE)
print("Meta:", META_FILE)
print("Verification:", VERIFICATION_FILE)
print("\nCELL 5: SUCCESS")
